# Prática — Regressão linear com uma variável: lucro de restaurantes
**Curso 1 · Semana 2 · Practice Lab — Andrew Ng**

Material de estudo independente, baseado em C1_W2_Linear_Regression.ipynb. O original é uma atividade avaliativa com compute_cost e compute_gradient em branco, além de funções auxiliares utils.py e public_tests.py e um dataset não anexado. Aqui explicamos e implementamos as fórmulas em um caderno didático separado, usando dados simulados para execução. Não é o notebook oficial de entrega nem uma reprodução dos testes do curso.

## 1. Problema

Uma franquia pretende abrir restaurantes em novas cidades. O dado de entrada x é a população em unidades de 10.000 habitantes. O alvo y é o lucro médio mensal em unidades de US$ 10.000. Um y negativo significa prejuízo. O original possui 97 exemplos.

Exemplo do material: x=6.1101 corresponde a 61.101 habitantes; y=17.592 corresponde a US$ 175.920; y=-2.6807 corresponde a prejuízo de US$ 26.807. x é a característica observada, y é o resultado real, e $\hat y$ é a previsão.

Importamos NumPy para operações numéricas, Matplotlib para gráficos, copy e math para estudar a estrutura da função de descida do gradiente original. Instale numpy e matplotlib, se necessário.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import math

# Base simulada: não é o dataset original de 97 cidades.
rng = np.random.default_rng(42)
x_train = np.linspace(5, 22, 97)
y_train = 1.15*x_train - 3.5 + rng.normal(0,2.0,97)
print("Tipo de x:", type(x_train))
print("Primeiros valores:", x_train[:5], y_train[:5])
print("Dimensões:", x_train.shape, y_train.shape)
print("Quantidade de exemplos:", len(x_train))

## 2. Conhecer os dados

type informa a classe do objeto; [:5] retorna os primeiros cinco elementos, sem incluir o índice 5. shape informa as dimensões de cada vetor; len(x_train) conta os exemplos. Ambos têm formato (97,), pois há apenas uma característica e um alvo por cidade. Cada par de valores com o mesmo índice representa uma cidade.

O gráfico de dispersão mostra população no eixo horizontal e lucro real no vertical. marker='x' define o símbolo; c='r' define vermelho. plt.show exibe a figura. A tendência pode ser crescente sem que os pontos estejam perfeitamente alinhados.

In [ ]:
plt.scatter(x_train,y_train,marker="x",c="r")
plt.xlabel("População (×10.000 habitantes)")
plt.ylabel("Lucro mensal (×US$ 10.000)")
plt.title("Lucro versus população — dados simulados")
plt.show()

## 3. Modelo e notação matemática

$$\hat y^{(i)}=f_{w,b}(x^{(i)})=wx^{(i)}+b$$

- $w$: peso ou inclinação da reta.
- $b$: intercepto, previsão quando x=0.
- $i$: índice do exemplo (linha), não uma potência. $x^{(0)}$ é o primeiro valor e $y^{(0)}$ é o primeiro alvo.
- $m$: quantidade de exemplos.
- $x$ e $y$ são fixos durante o treinamento; w e b mudam.

Exemplo manual: x=5, y=4, w=1, b=0. A previsão é 5 e o erro (previsão menos real) é +1. Se outra cidade tiver erro -1, a soma dos erros seria zero mesmo com ambas erradas. Por isso usamos a função de custo, que eleva cada erro ao quadrado antes de agregá-los.

O modelo estima associações, não demonstra que aumentar a população causará certo aumento de lucro.

## 4. Função de custo — compute_cost

$$J(w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}(wx^{(i)}+b-y^{(i)})^2$$

O $\hat y$ já está dentro da fórmula como wx+b. Primeiro calculamos a previsão, subtraímos o real, elevamos ao quadrado, somamos todos os exemplos e dividimos por 2m. O 2 facilita a derivação; o custo é metade do MSE.

O código a seguir é um exemplo independente em português de como implementar a matemática, não uma célula do notebook avaliativo original. O acumulador começa em zero; range(m) produz 0 até m-1; cada iteração adiciona um erro quadrático. A divisão ocorre depois de percorrer todos os exemplos.

**Cálculo à mão:** x=[2,3], y=[5,7], w=1, b=1. Previsões 3 e 4; erros -2 e -3; quadrados 4 e 9; custo (4+9)/(2×2)=3.25.

In [ ]:
def calcular_custo(x, y, w, b):
    m = x.shape[0]
    soma = 0.0
    for i in range(m):
        previsao = w*x[i] + b
        erro = previsao-y[i]
        soma += erro**2
    return soma/(2*m)

x_ex = np.array([2.,3.])
y_ex = np.array([5.,7.])
assert np.isclose(calcular_custo(x_ex,y_ex,1,1),3.25)
print("Custo do exemplo:", calcular_custo(x_ex,y_ex,1,1))
print("Custo do treino simulado para w=2 e b=1:", calcular_custo(x_train,y_train,2,1))

O notebook original espera custo 75.203 para w=2 e b=1 no dataset do curso. O valor acima usa dados simulados e não precisa coincidir. O assert é um teste independente com números conhecidos; não substitui public_tests.compute_cost_test da atividade oficial.

## 5. Derivadas parciais — compute_gradient

O gradiente informa a inclinação do custo em relação aos parâmetros. A derivada parcial de w mantém b fixo; a de b mantém w fixo.

Pela regra da cadeia, ao derivar $(wx+b-y)^2$ em relação a w, obtemos $2(wx+b-y)x$. O fator 2 cancela com o denominador 2m. Em relação a b, a derivada interna é 1, então não multiplicamos por x.

$$\frac{\partial J}{\partial w}=\frac{1}{m}\sum_{i=0}^{m-1}(wx^{(i)}+b-y^{(i)})x^{(i)}$$
$$\frac{\partial J}{\partial b}=\frac{1}{m}\sum_{i=0}^{m-1}(wx^{(i)}+b-y^{(i)})$$

Para o exemplo anterior, os erros são -2 e -3. A derivada de w é [(-2)×2+(-3)×3]/2=-6.5 e a de b=(-2-3)/2=-2.5. A função calcula essas duas médias; ainda não atualiza os parâmetros.

In [ ]:
def calcular_gradiente(x,y,w,b):
    m = len(x)
    soma_w = 0.0
    soma_b = 0.0
    for i in range(m):
        erro = w*x[i]+b-y[i]
        soma_w += erro*x[i]
        soma_b += erro
    return soma_w/m, soma_b/m

dw,db = calcular_gradiente(x_ex,y_ex,1,1)
assert np.allclose([dw,db],[-6.5,-2.5])
print("Gradientes manuais:",dw,db)
print("Gradientes iniciais simulados:", calcular_gradiente(x_train,y_train,0,0))
print("Gradientes de teste simulados:", calcular_gradiente(x_train,y_train,.2,.2))

No original, os valores esperados do gradiente em (0,0) são aproximadamente (-65.32884975,-5.83913505); em (0.2,0.2), (-47.41610118,-4.00717505). Esses são resultados específicos do dataset não anexado.

## 6. Descida do gradiente: função fornecida pelo professor

A atividade oficial já fornece gradient_descent. Ela recebe funções de custo e gradiente como argumentos, além de x, y, w inicial, b inicial, alpha e número de iterações. Assim a implementação da otimização fica separada da matemática específica de cada modelo.

$$w:=w-\alpha\frac{1}{m}\sum_i(wx^{(i)}+b-y^{(i)})x^{(i)}$$
$$b:=b-\alpha\frac{1}{m}\sum_i(wx^{(i)}+b-y^{(i)})$$

Com os gradientes -6.5 e -2.5, se alpha=0.1, o novo w=1.65 e b=1.25. Os dois gradientes vêm dos parâmetros anteriores; a atualização simultânea é fundamental. O sinal de menos move na direção oposta ao gradiente.

A versão didática abaixo conserva a estrutura da função pronta: copy.deepcopy evita modificar o parâmetro inicial quando mutável; J_history guarda custos; w_history registra pesos em intervalos; math.ceil arredonda para cima para definir a frequência de impressão. O limite de 100.000 registros é a proteção de memória do original. Acrescentamos verificação de custo finito e calculamos o custo corrente independentemente de guardar histórico, para evitar impressão de valor antigo após atingir o limite. Este é batch gradient descent: uma iteração usa todos os exemplos.

In [ ]:
def descida_do_gradiente(x,y,w_in,b_in,cost_function,gradient_function,alpha,num_iters):
    w = copy.deepcopy(w_in)
    b = b_in
    J_history = []
    w_history = []
    for i in range(num_iters):
        dj_dw,dj_db = gradient_function(x,y,w,b)
        w = w-alpha*dj_dw
        b = b-alpha*dj_db
        custo = cost_function(x,y,w,b)
        if not np.isfinite(custo):
            raise ValueError("O custo divergiu; reduza alpha.")
        if i < 100000:
            J_history.append(custo)
        if i % max(1,math.ceil(num_iters/10)) == 0:
            w_history.append(w)
            print(f"Iteração {i:4}: custo {custo:8.4f}")
    return w,b,J_history,w_history

w1,b1,_,_ = descida_do_gradiente(x_ex,y_ex,1,1,calcular_custo,calcular_gradiente,.1,1)
assert np.allclose([w1,b1],[1.65,1.25])
print("Após um passo:", w1,b1)

## 7. Treinar e verificar convergência

O original utiliza w=b=0, alpha=0.01 e 1500 iterações. Como a base simulada tem outra escala e distribuição, verificamos o limite de estabilidade antes de escolher alpha. Para a função quadrática, L é o maior autovalor da matriz A.T @ A/m, sendo A formada por x e uma coluna de 1. Uma taxa menor que 2/L garante estabilidade no caso quadrático. Usamos min(0.01,1/L) para manter uma taxa segura neste exemplo; isso pode diferir da aula.

O original registra w≈1.16636235 e b≈-3.63029143940436 após o treinamento oficial. Aqui aprendemos outros valores. A curva de custo deve cair com uma taxa adequada e tende a estabilizar; um custo baixo no treino não garante boas previsões em outras cidades.

In [ ]:
A = np.column_stack((x_train,np.ones(len(x_train))))
L = np.linalg.eigvalsh(A.T@A/len(A))[-1]
alpha = min(.01,1/L)
w,b,J_history,w_history = descida_do_gradiente(
    x_train,y_train,0.,0.,calcular_custo,calcular_gradiente,alpha,1500)
print("Taxa:",alpha,"w:",w,"b:",b)
plt.plot(J_history)
plt.xlabel("Iterações (custo após a atualização)")
plt.ylabel("J(w,b)")
plt.title("Convergência")
plt.show()

## 8. Previsão e visualização da reta

O original cria um vetor predicted com np.zeros(m) e preenche cada elemento em um for. Isso deixa evidente a relação entre a fórmula e o código. A forma vetorizada é w*x_train+b; a matemática é idêntica, mas NumPy executa operações sobre o vetor inteiro. np.allclose verifica a equivalência numérica. A reta usa os valores aprendidos de w,b e os pontos representam alvos do treino.

In [ ]:
predicted = np.zeros(len(x_train))
for i in range(len(x_train)):
    predicted[i] = w*x_train[i]+b
predicted_vetor = w*x_train+b
assert np.allclose(predicted,predicted_vetor)
plt.plot(x_train,predicted,label="Previsão")
plt.scatter(x_train,y_train,marker="x",label="Real")
plt.xlabel("População (×10.000)")
plt.ylabel("Lucro mensal (×US$ 10.000)")
plt.legend()
plt.show()

## 9. Prever lucro em cidades novas

O original pede cidades de 35.000 e 70.000 habitantes. Como a unidade do treino é 10.000 pessoas, as entradas são 3.5 e 7.0. Como y está em dezenas de milhares de dólares, multiplicamos a previsão por 10.000.

O notebook oficial registra resultados esperados de US$ 4.519,77 e US$ 45.342,45. Os valores abaixo são do modelo simulado. Observe que 3.5 está fora do intervalo de treino simulado (5 a 22): é extrapolação e deve ser interpretada com cautela.

In [ ]:
for populacao in [35000,70000]:
    entrada = populacao/10000
    previsao = (w*entrada+b)*10000
    print(f"População {populacao}: lucro mensal simulado previsto US$ {previsao:,.2f}")

## 10. Resumo e exercícios

| Etapa | O que faz |
|---|---|
| Previsão | wx+b estima o alvo |
| Erro | $\hat y-y$ mede o desvio de um exemplo |
| Custo | agrega erros quadráticos de todo o conjunto |
| Derivadas | indicam a inclinação do custo em w e b |
| Descida do gradiente | aplica as atualizações com alpha |
| Predição final | usa os parâmetros treinados em novos dados |

**Exercícios:** (1) Para x=3, w=2, b=1, y=8, qual previsão, erro e erro quadrático? (2) Para esse único exemplo, quais as derivadas? (3) Atualize w,b com alpha=0.1. (4) Por que o índice i não é uma potência? (5) Qual o propósito de J_history?

**Respostas:** (1) 7, -1, 1. (2) dw=-3, db=-1. (3) w=2.3, b=1.1. (4) Identifica a linha do dataset. (5) Monitorar a redução do custo e a convergência.

Fonte: Practice Lab: Linear Regression, Andrew Ng, DeepLearning.AI / Stanford Online, notebook fornecido pelo aluno. Use o notebook oficial e seus arquivos auxiliares para a avaliação oficial.